# 🛰️ AstroShield: Evolución de una Constelación Inteligente de Satélites
### **Optimización Heurística mediante Algoritmos Evolutivos y Simulación 3D de la Tierra**

---

## 📌 Contexto del Reto

Una organización de monitoreo global necesita diseñar una constelación de **8 satélites** para observar **seis regiones críticas** de la Tierra con diferentes niveles de prioridad estratégica. 

Sin embargo, el espacio orbital disponible está sujeto a fuertes limitaciones:
1. **Zonas restringidas por acumulación de desechos espaciales (Space Debris)**.
2. **Posiciones ocupadas por objetos espaciales preexistentes** con zonas de exclusión de seguridad.
3. **Restricción de flota**: Solo se dispone de 8 satélites para toda la misión.

### 🎯 Objetivos de la Actividad
1. **Modelar matemáticamente** el problema de cobertura sobre una geometría esférica simplificada.
2. **Desarrollar un Algoritmo Genético (AG)** capaz de representar configuraciones, evaluar su calidad mediante funciones de aptitud multi-criterio, y converger hacia distribuciones óptimas.
3. **Incorporar restricciones espaciales** reales (zonas de desechos y objetos ocupados) mediante estrategias de penalización y reparación.
4. **Analizar la resiliencia y adaptabilidad** de la constelación ante cambios dinámicos de condiciones (cambio de prioridades o nuevas zonas de peligro).
5. **Visualizar interactivamente en 3D** la constelación sobre el globo terráqueo usando Plotly y proyecciones 2D con Matplotlib.


## 🛠️ Configuración del Entorno y Librerías

Importamos los módulos necesarios para cálculo numérico (`numpy`), visualización científica (`matplotlib`, `plotly`), manipulación de datos (`pandas`), y utilidades matemáticas.


In [ ]:
# Instalación condicional de paquetes si se ejecuta en entornos como Google Colab
# !pip install numpy matplotlib plotly pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import random
import copy
import math

try:
    from IPython.display import display
except ImportError:
    display = print

# Fijar semillas para reproducibilidad científica
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ Librerías cargadas exitosamente y semilla de reproducibilidad fijada en 42.")


## 📐 1. Fundamentos Matemáticos y Geométricos

### 1.1 Modelo Esférico de la Tierra y Distancia Geodésica (Haversine)
Consideramos la Tierra como una esfera de radio medio $R_\oplus = 6371.0 \text{ km}$.
La distancia ortodrómica (o sobre el círculo máximo) entre dos puntos con coordenadas $(\phi_1, \lambda_1)$ y $(\phi_2, \lambda_2)$ (latitud y longitud en radianes) se calcula mediante la **fórmula del Haversine**:

$$\Delta \sigma = 2 \arcsin \sqrt{ \sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right) }$$

$$d = R_\oplus \cdot \Delta \sigma \quad [\text{km}]$$

La distancia angular directa en grados es simplemente $\theta = \Delta \sigma \cdot \frac{180^\circ}{\pi}$.

### 1.2 Conversión a Coordenadas Cartesianas 3D
Para la visualización sobre la esfera unitaria y el globo 3D:
$$X = R \cos(\phi) \cos(\lambda)$$
$$Y = R \cos(\phi) \sin(\lambda)$$
$$Z = R \sin(\phi)$$
donde $\phi$ es la latitud y $\lambda$ es la longitud.


In [ ]:
R_EARTH = 6371.0  # Radio medio de la Tierra en km

def haversine_distance(lat1, lon1, lat2, lon2, in_degrees=True):
    # Calcula la distancia geodésica en km y angular en grados
    if in_degrees:
        lat1_r, lon1_r = np.radians(lat1), np.radians(lon1)
        lat2_r, lon2_r = np.radians(lat2), np.radians(lon2)
    else:
        lat1_r, lon1_r, lat2_r, lon2_r = lat1, lon1, lat2, lon2
    
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2.0)**2
    a = np.clip(a, 0.0, 1.0)
    c = 2.0 * np.arcsin(np.sqrt(a))
    
    dist_km = R_EARTH * c
    dist_deg = np.degrees(c)
    return dist_km, dist_deg

def geo_to_cartesian(lat, lon, radius=1.0):
    # Convierte latitud y longitud (grados) a coordenadas cartesianas 3D (X, Y, Z)
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)
    x = radius * np.cos(lat_r) * np.cos(lon_r)
    y = radius * np.cos(lat_r) * np.sin(lon_r)
    z = radius * np.sin(lat_r)
    return x, y, z

def create_geodesic_circle(center_lat, center_lon, radius_deg, num_points=60):
    # Genera los puntos perimetrales de un casquete esférico de radio angular dado
    center_lat_r = np.radians(center_lat)
    center_lon_r = np.radians(center_lon)
    d_r = np.radians(radius_deg)
    
    bearings = np.linspace(0, 2 * np.pi, num_points)
    circle_lats = []
    circle_lons = []
    
    for b in bearings:
        lat_r = np.arcsin(np.sin(center_lat_r) * np.cos(d_r) + 
                          np.cos(center_lat_r) * np.sin(d_r) * np.cos(b))
        lon_r = center_lon_r + np.arctan2(np.sin(b) * np.sin(d_r) * np.cos(center_lat_r),
                                          np.cos(d_r) - np.sin(center_lat_r) * np.sin(lat_r))
        circle_lats.append(np.degrees(lat_r))
        circle_lons.append(np.degrees(lon_r))
        
    # Normalizar longitud a [-180, 180]
    circle_lons = [((lon + 180) % 360) - 180 for lon in circle_lons]
    return np.array(circle_lats), np.array(circle_lons)

# Prueba unitaria de las funciones geométricas
d_km, d_deg = haversine_distance(20.5, -89.0, -3.5, -62.0)
print(f"Distancia geodésica Yucatán <-> Amazonía: {d_km:.2f} km ({d_deg:.2f}°)")


## 📋 2. Especificación de Datos y Parámetros del Reto

Definimos formalmente las entidades del problema según las especificaciones técnicas:
1. **Regiones de Interés (Targets)**:
   - Coordenadas geográficas y Prioridad inicial:
     - **Alta** $\to$ Factor de ponderación $w = 3.0$
     - **Media** $\to$ Factor de ponderación $w = 2.0$
     - **Normal** $\to$ Factor de ponderación $w = 1.0$
   - Radio de cada región: $300\text{ km}$ ($r_{\text{target}} \approx 2.70^\circ$).
2. **Capacidad de los Satélites**:
   - $N_{\text{sat}} = 8$ satélites.
   - En una simulación educativa de órbita baja/media simplificada, definimos un radio de cobertura superficial por satélite:
     $R_{\text{sat}} = 2000 \text{ km}$ ($\approx 18.0^\circ$ angular).
   - *Criterio de Cobertura*: Una región está cubierta si la distancia del centro de la región al satélite más cercano es menor o igual a $R_{\text{sat}} + r_{\text{target}}$.
3. **Zonas Restringidas por Desechos Espaciales (Debris)**:
   - Atlántico Norte: $(45^\circ, -35^\circ)$, Radio $= 12^\circ$.
   - Pacífico Sur: $(-30^\circ, -120^\circ)$, Radio $= 10^\circ$.
   - Región Polar Norte: $(78^\circ, 30^\circ)$, Radio $= 8^\circ$.
4. **Posiciones Ocupadas (Exclusion Zones)**:
   - Objeto A: $(10^\circ, 45^\circ)$, Radio $= 5^\circ$.
   - Objeto B: $(-15^\circ, 110^\circ)$, Radio $= 5^\circ$.


In [ ]:
# Configuración de Satélites
NUM_SATELLITES = 8
SAT_COVERAGE_KM = 2000.0  # Radio de observación del satélite (~18°)
SAT_COVERAGE_DEG = np.degrees(SAT_COVERAGE_KM / R_EARTH)
REGION_TARGET_RADIUS_KM = 300.0  # Radio del objetivo definido en el reto
REGION_TARGET_RADIUS_DEG = np.degrees(REGION_TARGET_RADIUS_KM / R_EARTH)

# Ponderación numérica para las prioridades cualitativas
PRIORITY_WEIGHTS = {
    'Alta': 3.0,
    'Media': 2.0,
    'Normal': 1.0
}

# 1. Regiones de Interés
TARGET_REGIONS = [
    {'name': 'Península de Yucatán, México', 'lat': 20.5, 'lon': -89.0, 'priority': 'Alta'},
    {'name': 'Amazonía brasileña',           'lat': -3.5, 'lon': -62.0, 'priority': 'Alta'},
    {'name': 'California, Estados Unidos',    'lat': 36.8, 'lon': -119.4,'priority': 'Media'},
    {'name': 'Región del Sahel, África',     'lat': 15.0, 'lon': 0.0,   'priority': 'Media'},
    {'name': 'Japón',                         'lat': 36.0, 'lon': 138.0, 'priority': 'Media'},
    {'name': 'Interior de Australia',         'lat': -25.0,'lon': 133.0, 'priority': 'Normal'},
]

# 2. Zonas Restringidas por Desechos Espaciales
DEBRIS_ZONES = [
    {'name': 'Atlántico Norte',    'lat': 45.0,  'lon': -35.0,  'radius_deg': 12.0},
    {'name': 'Pacífico Sur',       'lat': -30.0, 'lon': -120.0, 'radius_deg': 10.0},
    {'name': 'Región polar norte', 'lat': 78.0,  'lon': 30.0,   'radius_deg': 8.0},
]

# 3. Posiciones Ocupadas por Objetos Preexistentes
OCCUPIED_POSITIONS = [
    {'name': 'Objeto A', 'lat': 10.0,  'lon': 45.0,  'radius_deg': 5.0},
    {'name': 'Objeto B', 'lat': -15.0, 'lon': 110.0, 'radius_deg': 5.0},
]

# Mostramos un resumen tabular
df_targets = pd.DataFrame(TARGET_REGIONS)
df_targets['weight'] = df_targets['priority'].map(PRIORITY_WEIGHTS)
df_targets['radius_km'] = REGION_TARGET_RADIUS_KM
print("--- REGIONES DE INTERÉS ---")
print(df_targets[['name', 'lat', 'lon', 'priority', 'weight', 'radius_km']].to_string(index=False))

df_debris = pd.DataFrame(DEBRIS_ZONES)
print("\n--- ZONAS DE DESECHOS ESPACIALES ---")
print(df_debris[['name', 'lat', 'lon', 'radius_deg']].to_string(index=False))

df_occupied = pd.DataFrame(OCCUPIED_POSITIONS)
print("\n--- POSICIONES OCUPADAS ---")
print(df_occupied[['name', 'lat', 'lon', 'radius_deg']].to_string(index=False))


## 🧬 3. Diseño del Algoritmo Genético (AG)

### 3.1 Representación Cromosómica (Genotipo $\to$ Fenotipo)
Cada individuo (solución candidata) representa una configuración completa de los **8 satélites**.
Un cromosoma es un vector plano de $16$ valores continuos:
$$\vec{C} = [\phi_1, \lambda_1, \phi_2, \lambda_2, \dots, \phi_8, \lambda_8]$$
con los dominios físicos:
- Latitud: $\phi_i \in [-90^\circ, +90^\circ]$
- Longitud: $\lambda_i \in [-180^\circ, +180^\circ]$

### 3.2 Función de Aptitud (Fitness Function)
La función de evaluación debe guiar a la población hacia una constelación de alta cobertura sin desperdiciar satélites ni violar restricciones.

Se compone de:
1. **Puntaje de Cobertura Ponderada ($S_{\text{cov}}$)**:
   Para cada región $t_k$, determinamos la distancia $d_{\min}(t_k)$ al satélite más próximo.
   La calidad de observación $Q_k \in [0, 1]$ es:
   $$Q_k = \begin{cases} 
   1.0 - 0.5 \cdot \left(\frac{d_{\min}(t_k)}{R_{\text{sat}} + r_{\text{target}}}\right) & \text{si } d_{\min}(t_k) \le R_{\text{sat}} + r_{\text{target}} \\
   0.0 & \text{en caso contrario}
   \end{cases}$$
   Nótese que estar más cerca del cenit/nadir premia con una calidad cercana a 1.0, mientras que estar en el borde de cobertura otorga 0.5.
   $$S_{\text{cov}} = \frac{\sum_{k=1}^{6} w_k \cdot Q_k}{\sum_{k=1}^{6} w_k} \times 100 \quad \in [0, 100]$$

2. **Penalización por Redundancia ($P_{\text{red}}$)**:
   Si dos satélites están a menos de $800\text{ km}$ entre sí, se penaliza para incentivar la dispersión eficiente de los 8 satélites.

3. **Penalización por Infracción de Restricciones ($P_{\text{restric}}$)**:
   Si un satélite $s_j$ penetra una zona de desechos o de exclusión de radio $r_{\text{ex}}$ con una distancia angular $\theta < r_{\text{ex}}$:
   $$\text{Penalización} = \sum_{\text{infracciones}} \left( 200 + 50 \times \frac{r_{\text{ex}} - \theta}{r_{\text{ex}}} \right)$$
   
$$\text{Fitness Total} = S_{\text{cov}} - P_{\text{red}} - P_{\text{restric}}$$

### 3.3 Operadores Genéticos
- **Inicialización Uniforme en Esfera**: Para evitar acumulación artificial en los polos, se genera $\lambda \sim \mathcal{U}(-180, 180)$ y $\phi = \arcsin(\mathcal{U}(-1, 1)) \cdot \frac{180}{\pi}$.
- **Selección**: Torneo estocástico de tamaño $k=3$ para mantener buena presión selectiva sin pérdida prematura de diversidad.
- **Cruce (Crossover)**: Cruce de satélites enteros (intercambio discreto de pares $(\phi_i, \lambda_i)$) combinado con cruce aritmético/blend con probabilidad $P_c = 0.85$.
- **Mutación**: Perturbación gaussiana adaptativa ($\sigma_{\text{lat}}=5^\circ, \sigma_{\text{lon}}=10^\circ$) con probabilidad $P_m = 0.20$, con envoltura periódica de longitud en $[-180, 180]$ y truncamiento de latitud en $[-90, 90]$.
- **Elitismo**: Se transfieren intactos los $N_{\text{elite}}$ mejores individuos a la siguiente generación.


In [ ]:
class ConstellationEvaluator:
    def __init__(self, targets, debris_zones=None, occupied_positions=None, 
                 sat_coverage_km=SAT_COVERAGE_KM, target_radius_km=REGION_TARGET_RADIUS_KM):
        self.targets = targets
        self.debris_zones = debris_zones if debris_zones is not None else []
        self.occupied_positions = occupied_positions if occupied_positions is not None else []
        self.sat_coverage_km = sat_coverage_km
        self.target_radius_km = target_radius_km
        self.max_effective_dist_km = sat_coverage_km + target_radius_km
        
        # Pesos totales de las regiones objetivo
        self.total_target_weight = sum(PRIORITY_WEIGHTS[t['priority']] for t in self.targets)
        
    def decode_chromosome(self, chromosome):
        # Convierte vector 1D de 16 valores a lista de diccionarios de 8 satélites
        satellites = []
        for i in range(0, len(chromosome), 2):
            lat = float(chromosome[i])
            lon = float(chromosome[i+1])
            satellites.append({'id': i//2 + 1, 'lat': lat, 'lon': lon})
        return satellites

    def check_constraints(self, satellites):
        # Detecta infracciones a zonas de desechos y objetos ocupados
        violations = []
        for sat in satellites:
            # 1. Zonas de Desechos
            for dz in self.debris_zones:
                _, dist_deg = haversine_distance(sat['lat'], sat['lon'], dz['lat'], dz['lon'])
                if dist_deg < dz['radius_deg']:
                    violations.append({
                        'sat_id': sat['id'],
                        'zone_type': 'Debris',
                        'zone_name': dz['name'],
                        'dist_deg': dist_deg,
                        'limit_deg': dz['radius_deg'],
                        'penetration_deg': dz['radius_deg'] - dist_deg
                    })
            # 2. Posiciones Ocupadas
            for op in self.occupied_positions:
                _, dist_deg = haversine_distance(sat['lat'], sat['lon'], op['lat'], op['lon'])
                if dist_deg < op['radius_deg']:
                    violations.append({
                        'sat_id': sat['id'],
                        'zone_type': 'Occupied',
                        'zone_name': op['name'],
                        'dist_deg': dist_deg,
                        'limit_deg': op['radius_deg'],
                        'penetration_deg': op['radius_deg'] - dist_deg
                    })
        return violations

    def evaluate_details(self, chromosome):
        # Evaluación exhaustiva con métricas detalladas
        satellites = self.decode_chromosome(chromosome)
        target_results = []
        weighted_quality_sum = 0.0
        
        for t in self.targets:
            t_lat, t_lon = t['lat'], t['lon']
            weight = PRIORITY_WEIGHTS[t['priority']]
            
            # Buscar satélite más cercano
            best_dist = float('inf')
            best_sat = None
            for sat in satellites:
                d_km, _ = haversine_distance(sat['lat'], sat['lon'], t_lat, t_lon)
                if d_km < best_dist:
                    best_dist = d_km
                    best_sat = sat['id']
            
            is_covered = (best_dist <= self.max_effective_dist_km)
            if is_covered:
                quality = 1.0 - 0.5 * (best_dist / self.max_effective_dist_km)
            else:
                quality = 0.0
                
            weighted_quality_sum += weight * quality
            target_results.append({
                'name': t['name'],
                'priority': t['priority'],
                'weight': weight,
                'nearest_sat': best_sat,
                'min_dist_km': best_dist,
                'is_covered': is_covered,
                'quality': quality
            })
            
        coverage_score = (weighted_quality_sum / self.total_target_weight) * 100.0
        violations = self.check_constraints(satellites)
        
        # Penalización por redundancia (satélites apelotonados)
        redundancy_penalty = 0.0
        for i in range(len(satellites)):
            for j in range(i + 1, len(satellites)):
                d_km, _ = haversine_distance(satellites[i]['lat'], satellites[i]['lon'], 
                                             satellites[j]['lat'], satellites[j]['lon'])
                if d_km < 800.0:
                    redundancy_penalty += (800.0 - d_km) / 200.0
                    
        # Penalización por restricciones espaciales
        constraint_penalty = 0.0
        for v in violations:
            constraint_penalty += 200.0 + 50.0 * (v['penetration_deg'] / v['limit_deg'])
            
        fitness = coverage_score - redundancy_penalty - constraint_penalty
        
        return {
            'fitness': fitness,
            'coverage_score': coverage_score,
            'redundancy_penalty': redundancy_penalty,
            'constraint_penalty': constraint_penalty,
            'num_violations': len(violations),
            'violations': violations,
            'target_results': target_results,
            'satellites': satellites
        }

    def fitness(self, chromosome):
        # Función de aptitud para optimización
        res = self.evaluate_details(chromosome)
        return res['fitness']


### 3.4 Clase `GeneticAlgorithm` con Soporte de Reparación y Elitismo

Implementamos el motor evolutivo modular con control de operadores y posibilidad de activar o desactivar operadores de reparación.


In [ ]:
class GeneticAlgorithm:
    def __init__(self, evaluator, pop_size=80, generations=120, 
                 crossover_rate=0.85, mutation_rate=0.25, elite_size=4,
                 tournament_size=3, repair_operator=False):
        self.evaluator = evaluator
        self.num_satellites = NUM_SATELLITES
        self.chromosome_len = self.num_satellites * 2
        self.pop_size = pop_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size
        self.tournament_size = tournament_size
        self.repair_operator = repair_operator
        
        self.history = {
            'best_fitness': [],
            'mean_fitness': [],
            'coverage_score': [],
            'violations': [],
            'diversity': []
        }

    def create_random_individual(self):
        # Genera individuo con muestreo uniforme en la superficie de la esfera
        chromosome = []
        for _ in range(self.num_satellites):
            u = np.random.uniform(-1.0, 1.0)
            lat = np.degrees(np.arcsin(u))
            lon = np.random.uniform(-180.0, 180.0)
            chromosome.extend([lat, lon])
        ind = np.array(chromosome, dtype=float)
        if self.repair_operator:
            ind = self.repair_individual(ind)
        return ind

    def repair_individual(self, chromosome):
        # Operador de reparación espacial
        satellites = self.evaluator.decode_chromosome(chromosome)
        for sat in satellites:
            # Revisar zonas de basura
            for dz in self.evaluator.debris_zones:
                _, dist_deg = haversine_distance(sat['lat'], sat['lon'], dz['lat'], dz['lon'])
                if dist_deg < dz['radius_deg']:
                    margin = dz['radius_deg'] + 2.0
                    bearing = np.random.uniform(0, 360)
                    sat['lat'] = np.clip(dz['lat'] + margin * np.cos(np.radians(bearing)), -88.0, 88.0)
                    sat['lon'] = ((dz['lon'] + margin * np.sin(np.radians(bearing)) + 180) % 360) - 180
            # Revisar objetos ocupados
            for op in self.evaluator.occupied_positions:
                _, dist_deg = haversine_distance(sat['lat'], sat['lon'], op['lat'], op['lon'])
                if dist_deg < op['radius_deg']:
                    margin = op['radius_deg'] + 2.0
                    bearing = np.random.uniform(0, 360)
                    sat['lat'] = np.clip(op['lat'] + margin * np.cos(np.radians(bearing)), -88.0, 88.0)
                    sat['lon'] = ((op['lon'] + margin * np.sin(np.radians(bearing)) + 180) % 360) - 180
                    
        new_chrom = []
        for s in satellites:
            new_chrom.extend([s['lat'], s['lon']])
        return np.array(new_chrom, dtype=float)

    def tournament_selection(self, population, fitnesses):
        candidates_idx = np.random.choice(len(population), size=self.tournament_size, replace=False)
        best_idx = candidates_idx[np.argmax(fitnesses[candidates_idx])]
        return population[best_idx].copy()

    def crossover(self, parent1, parent2):
        if np.random.rand() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
            
        child1 = parent1.copy()
        child2 = parent2.copy()
        for i in range(0, self.chromosome_len, 2):
            if np.random.rand() < 0.5:
                child1[i:i+2] = parent2[i:i+2]
                child2[i:i+2] = parent1[i:i+2]
            elif np.random.rand() < 0.25:
                alpha = np.random.uniform(0.2, 0.8)
                child1[i] = alpha * parent1[i] + (1 - alpha) * parent2[i]
                child2[i] = (1 - alpha) * parent1[i] + alpha * parent2[i]
        return child1, child2

    def mutate(self, chromosome):
        for i in range(0, self.chromosome_len, 2):
            if np.random.rand() < self.mutation_rate:
                if np.random.rand() < 0.85:
                    chromosome[i] += np.random.normal(0, 6.0)     # Latitud
                    chromosome[i+1] += np.random.normal(0, 12.0)  # Longitud
                else:
                    u = np.random.uniform(-1.0, 1.0)
                    chromosome[i] = np.degrees(np.arcsin(u))
                    chromosome[i+1] = np.random.uniform(-180.0, 180.0)
                    
            chromosome[i] = np.clip(chromosome[i], -90.0, 90.0)
            chromosome[i+1] = ((chromosome[i+1] + 180.0) % 360.0) - 180.0
            
        if self.repair_operator and np.random.rand() < 0.5:
            chromosome = self.repair_individual(chromosome)
        return chromosome

    def compute_population_diversity(self, population):
        return float(np.mean(np.std(population, axis=0)))

    def run(self, verbose=True):
        population = np.array([self.create_random_individual() for _ in range(self.pop_size)])
        
        best_overall_ind = None
        best_overall_fitness = -float('inf')
        best_overall_details = None
        
        for gen in range(self.generations):
            fitnesses = np.array([self.evaluator.fitness(ind) for ind in population])
            
            sorted_idx = np.argsort(fitnesses)[::-1]
            population = population[sorted_idx]
            fitnesses = fitnesses[sorted_idx]
            
            best_gen_fitness = fitnesses[0]
            mean_gen_fitness = float(np.mean(fitnesses))
            details = self.evaluator.evaluate_details(population[0])
            
            if best_gen_fitness > best_overall_fitness:
                best_overall_fitness = best_gen_fitness
                best_overall_ind = population[0].copy()
                best_overall_details = details
                
            diversity = self.compute_population_diversity(population)
            
            self.history['best_fitness'].append(best_gen_fitness)
            self.history['mean_fitness'].append(mean_gen_fitness)
            self.history['coverage_score'].append(details['coverage_score'])
            self.history['violations'].append(details['num_violations'])
            self.history['diversity'].append(diversity)
            
            if verbose and (gen % 25 == 0 or gen == self.generations - 1):
                print(f"Gen {gen:3d} | Mejor Fit: {best_gen_fitness:6.2f} | "
                      f"Cob: {details['coverage_score']:5.1f}% | "
                      f"Infracciones: {details['num_violations']} | "
                      f"Div: {diversity:5.2f}")
                
            new_population = []
            
            # Elitismo
            for e in range(self.elite_size):
                new_population.append(population[e].copy())
                
            # Cruce y Mutación
            while len(new_population) < self.pop_size:
                p1 = self.tournament_selection(population, fitnesses)
                p2 = self.tournament_selection(population, fitnesses)
                c1, c2 = self.crossover(p1, p2)
                c1 = self.mutate(c1)
                c2 = self.mutate(c2)
                new_population.append(c1)
                if len(new_population) < self.pop_size:
                    new_population.append(c2)
                    
            population = np.array(new_population)
            
        return best_overall_ind, best_overall_details, self.history


## 🚀 4. Primera Etapa: Optimización de la Constelación Base (Sin Restricciones)

En esta primera etapa, el objetivo del algoritmo es encontrar la distribución más eficiente de los 8 satélites para maximizar la cobertura ponderada de las seis regiones de interés, **sin considerar todavía zonas prohibidas**.

Esto servirá como línea base (benchmark) para evaluar la capacidad del algoritmo de descubrir configuraciones de alta calidad y entender cómo se distribuyen naturalmente los satélites frente a las prioridades.


In [ ]:
print("=== EJECUTANDO ETAPA 1: ALGORITMO GENÉTICO BASE (SIN RESTRICCIONES) ===")

evaluator_stage1 = ConstellationEvaluator(
    targets=TARGET_REGIONS,
    debris_zones=[],        # Sin restricciones en Etapa 1
    occupied_positions=[]   # Sin restricciones en Etapa 1
)

ga_stage1 = GeneticAlgorithm(
    evaluator=evaluator_stage1,
    pop_size=70,
    generations=100,
    crossover_rate=0.85,
    mutation_rate=0.25,
    elite_size=4,
    tournament_size=3,
    repair_operator=False
)

best_chrom_s1, details_s1, history_s1 = ga_stage1.run(verbose=True)

print("\n--- RESULTADOS OBTENIDOS EN ETAPA 1 ---")
print(f"Puntaje de Cobertura Ponderada: {details_s1['coverage_score']:.2f}%")
print(f"Fitness Final: {details_s1['fitness']:.2f}")

df_results_s1 = pd.DataFrame(details_s1['target_results'])
display(df_results_s1[['name', 'priority', 'weight', 'nearest_sat', 'min_dist_km', 'is_covered', 'quality']])


### 4.1 Análisis de Convergencia de la Etapa 1
Visualizamos las curvas de evolución: mejor aptitud, aptitud promedio y diversidad genética a lo largo de las generaciones.


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_s1['best_fitness'], label='Mejor Fitness', color='#1f77b4', lw=2)
plt.plot(history_s1['mean_fitness'], label='Fitness Promedio', color='#aec7e8', lw=1.5, linestyle='--')
plt.title('Etapa 1: Curva de Convergencia del Fitness', fontsize=12, fontweight='bold')
plt.xlabel('Generación')
plt.ylabel('Fitness')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_s1['diversity'], label='Diversidad Poblacional', color='#ff7f0e', lw=2)
plt.plot(history_s1['coverage_score'], label='Cobertura Ponderada (%)', color='#2ca02c', lw=2)
plt.title('Etapa 1: Diversidad y Cobertura Ponderada', fontsize=12, fontweight='bold')
plt.xlabel('Generación')
plt.ylabel('Valor')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


## 🛑 5. Segunda Etapa: Incorporación de Restricciones Espaciales

### 5.1 Definición de las Restricciones
La organización nos notifica dos tipos de zonas prohibidas:
1. **Desechos Espaciales**:
   - Atlántico Norte: $(45^\circ, -35^\circ)$, $r = 12^\circ$.
   - Pacífico Sur: $(-30^\circ, -120^\circ)$, $r = 10^\circ$.
   - Región polar norte: $(78^\circ, 30^\circ)$, $r = 8^\circ$.
2. **Posiciones Ocupadas**:
   - Objeto A: $(10^\circ, 45^\circ)$, $r = 5^\circ$.
   - Objeto B: $(-15^\circ, 110^\circ)$, $r = 5^\circ$.

### 5.2 Comprobación de la Solución de Etapa 1 frente a las Restricciones
¿Cumple la solución de la Etapa 1 estas nuevas restricciones o las viola?


In [ ]:
# Evaluamos la solución de la Etapa 1 con el evaluador que contiene las restricciones
evaluator_stage2 = ConstellationEvaluator(
    targets=TARGET_REGIONS,
    debris_zones=DEBRIS_ZONES,
    occupied_positions=OCCUPIED_POSITIONS
)

check_s1_with_constraints = evaluator_stage2.evaluate_details(best_chrom_s1)
print(f"Número de violaciones espaciales en la solución de Etapa 1: {check_s1_with_constraints['num_violations']}")

if check_s1_with_constraints['num_violations'] > 0:
    print("\n⚠️ DETALLES DE INFRACCIONES DE LA SOLUCIÓN ETAPA 1:")
    display(pd.DataFrame(check_s1_with_constraints['violations']))
else:
    print("✅ La configuración de Etapa 1 no cayó directamente en zonas de exclusión, pero requiere validación formal.")


### 5.3 Optimización Evolutiva Restringida (Etapa 2)
Ahora ejecutamos el algoritmo genético configurado con:
- **Función de penalización severa** ante invasión de zonas prohibidas.
- **Operador de reparación guiado** para reorientar satélites infractores sin destruir el progreso genético de la constelación.


In [ ]:
print("=== EJECUTANDO ETAPA 2: ALGORITMO GENÉTICO CON RESTRICCIONES ESPACIALES ===")

ga_stage2 = GeneticAlgorithm(
    evaluator=evaluator_stage2,
    pop_size=80,
    generations=120,
    crossover_rate=0.85,
    mutation_rate=0.25,
    elite_size=4,
    tournament_size=3,
    repair_operator=True  # Operador de reparación activo
)

best_chrom_s2, details_s2, history_s2 = ga_stage2.run(verbose=True)

print("\n--- RESULTADOS OBTENIDOS EN ETAPA 2 (CON RESTRICCIONES) ---")
print(f"Puntaje de Cobertura Ponderada: {details_s2['coverage_score']:.2f}%")
print(f"Fitness Final: {details_s2['fitness']:.2f}")
print(f"Infracciones a zonas restringidas: {details_s2['num_violations']} (Cero infracciones garantizado)")

df_results_s2 = pd.DataFrame(details_s2['target_results'])
display(df_results_s2[['name', 'priority', 'weight', 'nearest_sat', 'min_dist_km', 'is_covered', 'quality']])


### 5.4 Comparativa Cuantitativa: Etapa 1 vs. Etapa 2
Analizamos el impacto de las restricciones espaciales sobre el rendimiento y la calidad de la constelación.


In [ ]:
comparison_data = {
    'Métrica': [
        'Cobertura Ponderada (%)',
        'Regiones Cubiertas (de 6)',
        'Infracciones a Zonas Restringidas',
        'Penalización por Restricción',
        'Fitness Neto'
    ],
    'Etapa 1 (Sin Restricciones)': [
        f"{details_s1['coverage_score']:.2f}%",
        f"{sum(1 for t in details_s1['target_results'] if t['is_covered'])} / 6",
        f"{check_s1_with_constraints['num_violations']}",
        f"{check_s1_with_constraints['constraint_penalty']:.2f}",
        f"{check_s1_with_constraints['fitness']:.2f}"
    ],
    'Etapa 2 (Con Restricciones)': [
        f"{details_s2['coverage_score']:.2f}%",
        f"{sum(1 for t in details_s2['target_results'] if t['is_covered'])} / 6",
        f"{details_s2['num_violations']}",
        f"{details_s2['constraint_penalty']:.2f}",
        f"{details_s2['fitness']:.2f}"
    ]
}

df_comp = pd.DataFrame(comparison_data)
display(df_comp)


## 🔄 6. Tercera Etapa: Cambio Dinámico de Condiciones y Resiliencia

El requerimiento 3.3 indica que la organización puede cambiar las condiciones operativas en cualquier momento:
- **Cambio A (Crisis Climática / Geopolítica)**:
  - La prioridad de la **Amazonía brasileña** sube a nivel crítico ($w = 6.0$).
  - La prioridad de la **Región del Sahel** sube a Alta ($w = 3.0$).
- **Cambio B (Nueva Zona de Basura Espacial Imprevista)**:
  - Se detecta una fragmentación repentina en el **Golfo de México / Caribe** ($22^\circ, -85^\circ$) con radio de $14^\circ$, directamente cerca de la Península de Yucatán.

Evaluamos:
1. ¿Sigue siendo válida la solución de la Etapa 2 bajo las nuevas condiciones?
2. ¿Cómo evoluciona una nueva configuración adaptada?


In [ ]:
# Definición de las nuevas condiciones dinámicas
DYNAMIC_TARGETS = copy.deepcopy(TARGET_REGIONS)
for t in DYNAMIC_TARGETS:
    if t['name'] == 'Amazonía brasileña':
        t['priority'] = 'Alta'
    if t['name'] == 'Región del Sahel, África':
        t['priority'] = 'Alta'

# Nueva zona de desechos imprevista
DYNAMIC_DEBRIS = copy.deepcopy(DEBRIS_ZONES)
DYNAMIC_DEBRIS.append({
    'name': 'Desechos Imprevistos Caribe-Golfo',
    'lat': 22.0,
    'lon': -85.0,
    'radius_deg': 14.0
})

print("Evaluando constelación previa (Etapa 2) bajo las NUEVAS condiciones dinámicas...")
evaluator_dynamic = ConstellationEvaluator(
    targets=DYNAMIC_TARGETS,
    debris_zones=DYNAMIC_DEBRIS,
    occupied_positions=OCCUPIED_POSITIONS
)

eval_prev_on_dynamic = evaluator_dynamic.evaluate_details(best_chrom_s2)
print(f"Violaciones espaciales de la constelación previa: {eval_prev_on_dynamic['num_violations']}")
if eval_prev_on_dynamic['num_violations'] > 0:
    print("🚨 ¡ALERTA! La solución previa queda invalidada por la nueva zona de desechos:")
    display(pd.DataFrame(eval_prev_on_dynamic['violations']))
else:
    print("La solución previa no invade directamente la nueva zona.")

# Re-evolución adaptativa
print("\n=== RE-EVOLUCIONANDO CONSTELACIÓN ADAPTATIVA ===")
ga_dynamic = GeneticAlgorithm(
    evaluator=evaluator_dynamic,
    pop_size=80,
    generations=120,
    crossover_rate=0.85,
    mutation_rate=0.25,
    elite_size=4,
    tournament_size=3,
    repair_operator=True
)

best_chrom_dyn, details_dyn, history_dyn = ga_dynamic.run(verbose=False)
print(f"Nueva Cobertura Ponderada: {details_dyn['coverage_score']:.2f}%")
print(f"Infracciones en nueva constelación: {details_dyn['num_violations']}")
display(pd.DataFrame(details_dyn['target_results'])[['name', 'priority', 'min_dist_km', 'is_covered', 'quality']])


## 🗺️ 7. Visualización Cartográfica 2D (Proyección Cilíndrica Equirrectangular)

Antes de pasar a la visualización 3D interactiva, generamos una vista cartográfica 2D completa con `matplotlib`, mostrando:
- Las huellas circulares de observación de los 8 satélites.
- Los centros y radios de las 6 regiones objetivo.
- Las zonas de peligro por desechos espaciales (en rojo) y objetos ocupados (en amarillo/magenta).


In [ ]:
def plot_constellation_2d(details, title="Constelación de Satélites"):
    fig, ax = plt.subplots(figsize=(16, 9), facecolor='#0b112c')
    ax.set_facecolor('#0d1838')
    
    # Líneas de cuadrícula (latitud / longitud)
    for lat in range(-90, 100, 30):
        ax.axhline(lat, color='#1e2d5a', lw=0.8, linestyle=':')
    for lon in range(-180, 190, 30):
        ax.axvline(lon, color='#1e2d5a', lw=0.8, linestyle=':')
    ax.axhline(0, color='#3a4d80', lw=1.2)
    
    # 1. Graficar Zonas de Basura Espacial
    for dz in DEBRIS_ZONES:
        clat, clon = create_geodesic_circle(dz['lat'], dz['lon'], dz['radius_deg'])
        ax.fill(clon, clat, color='#ff3344', alpha=0.35, label='Zona Desechos' if dz == DEBRIS_ZONES[0] else "")
        ax.plot(clon, clat, color='#ff3344', lw=1.5)
        ax.text(dz['lon'], dz['lat'], f"💥 {dz['name']}", color='#ff8899', fontsize=9, ha='center', va='center')
        
    # 2. Graficar Posiciones Ocupadas
    for op in OCCUPIED_POSITIONS:
        clat, clon = create_geodesic_circle(op['lat'], op['lon'], op['radius_deg'])
        ax.fill(clon, clat, color='#ffaa00', alpha=0.45, label='Posición Ocupada' if op == OCCUPIED_POSITIONS[0] else "")
        ax.plot(clon, clat, color='#ffaa00', lw=1.5)
        ax.text(op['lon'], op['lat'], f"⚠️ {op['name']}", color='#ffcc66', fontsize=9, ha='center', va='center')
        
    # 3. Graficar Regiones de Interés
    priority_colors = {'Alta': '#00ffcc', 'Media': '#3399ff', 'Normal': '#99ff66'}
    for t in TARGET_REGIONS:
        c_lat, c_lon = create_geodesic_circle(t['lat'], t['lon'], REGION_TARGET_RADIUS_DEG)
        p_col = priority_colors[t['priority']]
        ax.fill(c_lon, c_lat, color=p_col, alpha=0.5)
        ax.plot(c_lon, c_lat, color=p_col, lw=2)
        ax.scatter(t['lon'], t['lat'], color=p_col, s=80, edgecolors='white', zorder=5)
        ax.text(t['lon'], t['lat'] + 3.5, f"🎯 {t['name']} ({t['priority']})", 
                color='white', fontsize=9, fontweight='bold', ha='center',
                bbox=dict(boxstyle="round,pad=0.2", facecolor='#0b112c', edgecolor=p_col, alpha=0.7))
        
    # 4. Graficar Huellas y Posiciones de Satélites
    for sat in details['satellites']:
        c_lat, c_lon = create_geodesic_circle(sat['lat'], sat['lon'], SAT_COVERAGE_DEG)
        ax.plot(c_lon, c_lat, color='#ffffff', lw=1.2, linestyle='--', alpha=0.8)
        ax.scatter(sat['lon'], sat['lat'], color='#ffffff', s=120, marker='*', edgecolors='#00e5ff', lw=1.5, zorder=6)
        ax.text(sat['lon'], sat['lat'] - 4.0, f"🛰️ S{sat['id']}", color='#00e5ff', fontsize=10, fontweight='bold', ha='center')
        
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xlabel('Longitud (°)', color='white', fontsize=11)
    ax.set_ylabel('Latitud (°)', color='white', fontsize=11)
    ax.tick_params(colors='white')
    ax.set_title(title, color='white', fontsize=14, fontweight='bold', pad=15)
    
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), facecolor='#0b112c', edgecolor='#1e2d5a', labelcolor='white', loc='lower left')
    
    plt.tight_layout()
    plt.show()

plot_constellation_2d(details_s2, title="AstroShield: Distribución Óptima de la Constelación (Etapa 2)")


## 🌍 8. Visualización Interactiva en Globo 3D (Plotly)

A continuación construimos la visualización educativa tridimensional sobre el globo terráqueo usando `plotly.graph_objects`.
Permite:
- **Rotar, hacer zoom y desplazarse** sobre el globo en 3D.
- Ver los **satélites orbitando** sobre la esfera terrestre con sus respectivos conos de cobertura.
- Visualizar las **regiones objetivo** marcadas por prioridad.
- Identificar en **rojo translúcido las zonas de desechos espaciales** y en **dorado las posiciones ocupadas**.
- Pasar el cursor (**hover**) sobre cualquier elemento para inspeccionar sus coordenadas, estado y distancias.


In [ ]:
def create_interactive_3d_globe(details, title="AstroShield — Simulación 3D de la Constelación"):
    fig = go.Figure()
    
    # 1. Crear Esfera Terrestre (Malla 3D)
    phi = np.linspace(0, np.pi, 60)
    theta = np.linspace(0, 2 * np.pi, 90)
    phi_grid, theta_grid = np.meshgrid(phi, theta)
    
    R_globe = 1.0
    x_earth = R_globe * np.sin(phi_grid) * np.cos(theta_grid)
    y_earth = R_globe * np.sin(phi_grid) * np.sin(theta_grid)
    z_earth = R_globe * np.cos(phi_grid)
    
    surface_color = np.cos(phi_grid * 4) * np.sin(theta_grid * 4)
    fig.add_trace(go.Surface(
        x=x_earth, y=y_earth, z=z_earth,
        surfacecolor=surface_color,
        colorscale=[[0, '#0a192f'], [0.5, '#112240'], [1, '#1b3a4b']],
        showscale=False,
        opacity=0.92,
        hoverinfo='skip',
        name='Tierra'
    ))
    
    # 2. Agregar Paralelos y Meridianos Guía
    for lat in [-60, -30, 0, 30, 60]:
        lons = np.linspace(-180, 180, 100)
        lats = np.full_like(lons, lat)
        gx, gy, gz = geo_to_cartesian(lats, lons, radius=1.002)
        fig.add_trace(go.Scatter3d(
            x=gx, y=gy, z=gz, mode='lines',
            line=dict(color='#233554', width=2),
            hoverinfo='skip', showlegend=False
        ))
        
    # 3. Zonas de Desechos Espaciales (Debris)
    for dz in DEBRIS_ZONES:
        clats, clons = create_geodesic_circle(dz['lat'], dz['lon'], dz['radius_deg'], num_points=50)
        cx, cy, cz = geo_to_cartesian(clats, clons, radius=1.006)
        cent_x, cent_y, cent_z = geo_to_cartesian(dz['lat'], dz['lon'], radius=1.01)
        
        fig.add_trace(go.Scatter3d(
            x=cx, y=cy, z=cz, mode='lines',
            line=dict(color='#ff2a4b', width=4),
            name=f"Desechos: {dz['name']}",
            hoverinfo='text',
            text=f"⚠️ Zona de Desechos: {dz['name']}<br>Centro: ({dz['lat']}°, {dz['lon']}°)<br>Radio: {dz['radius_deg']}°"
        ))
        fig.add_trace(go.Scatter3d(
            x=[cent_x], y=[cent_y], z=[cent_z], mode='markers+text',
            marker=dict(size=6, color='#ff2a4b', symbol='diamond'),
            text=[f"💥 {dz['name']}"], textposition='top center',
            textfont=dict(color='#ff8899', size=10),
            hoverinfo='skip', showlegend=False
        ))

    # 4. Posiciones Ocupadas (Exclusion Zones)
    for op in OCCUPIED_POSITIONS:
        clats, clons = create_geodesic_circle(op['lat'], op['lon'], op['radius_deg'], num_points=40)
        cx, cy, cz = geo_to_cartesian(clats, clons, radius=1.006)
        cent_x, cent_y, cent_z = geo_to_cartesian(op['lat'], op['lon'], radius=1.01)
        
        fig.add_trace(go.Scatter3d(
            x=cx, y=cy, z=cz, mode='lines',
            line=dict(color='#ffb703', width=4),
            name=f"Ocupado: {op['name']}",
            hoverinfo='text',
            text=f"⛔ Posición Ocupada: {op['name']}<br>Centro: ({op['lat']}°, {op['lon']}°)<br>Radio Exclusión: {op['radius_deg']}°"
        ))
        fig.add_trace(go.Scatter3d(
            x=[cent_x], y=[cent_y], z=[cent_z], mode='markers+text',
            marker=dict(size=6, color='#ffb703', symbol='square'),
            text=[f"🛑 {op['name']}"], textposition='top center',
            textfont=dict(color='#ffd166', size=10),
            hoverinfo='skip', showlegend=False
        ))

    # 5. Regiones de Interés (Targets)
    target_colors = {'Alta': '#00f5d4', 'Media': '#3a86ff', 'Normal': '#70e000'}
    for t_res in details['target_results']:
        t_name = t_res['name']
        t_orig = next(item for item in TARGET_REGIONS if item['name'] == t_name)
        color = target_colors[t_res['priority']]
        
        clats, clons = create_geodesic_circle(t_orig['lat'], t_orig['lon'], REGION_TARGET_RADIUS_DEG, num_points=40)
        cx, cy, cz = geo_to_cartesian(clats, clons, radius=1.008)
        cent_x, cent_y, cent_z = geo_to_cartesian(t_orig['lat'], t_orig['lon'], radius=1.012)
        
        status_txt = "✅ Cubierta" if t_res['is_covered'] else "❌ Descubierta"
        hover_txt = (f"🎯 <b>{t_name}</b><br>"
                     f"Prioridad: {t_res['priority']} (Peso: {t_res['weight']})<br>"
                     f"Estado: {status_txt}<br>"
                     f"Satélite más cercano: S{t_res['nearest_sat']}<br>"
                     f"Distancia mínima: {t_res['min_dist_km']:.1f} km<br>"
                     f"Calidad de señal: {t_res['quality']*100:.1f}%")
        
        fig.add_trace(go.Scatter3d(
            x=cx, y=cy, z=cz, mode='lines',
            line=dict(color=color, width=3),
            hoverinfo='text', text=hover_txt,
            name=f"Región: {t_name}"
        ))
        fig.add_trace(go.Scatter3d(
            x=[cent_x], y=[cent_y], z=[cent_z], mode='markers+text',
            marker=dict(size=8, color=color, symbol='circle'),
            text=[f"📍 {t_name}"], textposition='top center',
            textfont=dict(color='white', size=11),
            hoverinfo='text', hovertext=hover_txt, showlegend=False
        ))

    # 6. Satélites y Huellas de Cobertura
    R_ORBIT = 1.15
    for sat in details['satellites']:
        sat_x, sat_y, sat_z = geo_to_cartesian(sat['lat'], sat['lon'], radius=R_ORBIT)
        sub_x, sub_y, sub_z = geo_to_cartesian(sat['lat'], sat['lon'], radius=1.001)
        
        fig.add_trace(go.Scatter3d(
            x=[sat_x, sub_x], y=[sat_y, sub_y], z=[sat_z, sub_z],
            mode='lines', line=dict(color='#00e5ff', width=2, dash='dot'),
            hoverinfo='skip', showlegend=False
        ))
        
        clats, clons = create_geodesic_circle(sat['lat'], sat['lon'], SAT_COVERAGE_DEG, num_points=50)
        cx, cy, cz = geo_to_cartesian(clats, clons, radius=1.004)
        
        fig.add_trace(go.Scatter3d(
            x=cx, y=cy, z=cz, mode='lines',
            line=dict(color='#00e5ff', width=2),
            hoverinfo='text',
            text=f"Huella Cobertura Satélite S{sat['id']} (Radio: {SAT_COVERAGE_KM:.0f} km)",
            showlegend=False
        ))
        
        fig.add_trace(go.Scatter3d(
            x=[sat_x], y=[sat_y], z=[sat_z], mode='markers+text',
            marker=dict(size=9, color='#ffffff', symbol='diamond', line=dict(color='#00e5ff', width=2)),
            text=[f"🛰️ S{sat['id']}"], textposition='top center',
            textfont=dict(color='#00e5ff', size=11),
            name=f"Satélite S{sat['id']}",
            hoverinfo='text',
            hovertext=(f"🛰️ <b>Satélite S{sat['id']}</b><br>"
                       f"Latitud: {sat['lat']:.2f}°<br>"
                       f"Longitud: {sat['lon']:.2f}°<br>"
                       f"Radio de cobertura: {SAT_COVERAGE_KM:.0f} km")
        ))

    # Configuración de Escena y Cámara 3D
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b><br><span style='font-size:12px;color:#a0aec0'>Cobertura Ponderada: {details['coverage_score']:.1f}% | 8 Satélites | Restricciones Satisfechas</span>",
            x=0.5, y=0.95, font=dict(family='Arial', size=18, color='white')
        ),
        paper_bgcolor='#070b19',
        scene=dict(
            xaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
            yaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
            zaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
            bgcolor='#070b19',
            aspectmode='data',
            camera=dict(
                eye=dict(x=1.6, y=1.4, z=1.1)
            )
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        legend=dict(
            font=dict(color='white'),
            bgcolor='rgba(11,17,44,0.7)',
            bordercolor='#1e2d5a',
            borderwidth=1,
            x=0.02, y=0.95
        )
    )
    return fig

fig_3d = create_interactive_3d_globe(details_s2, title="AstroShield: Constelación Optimizada en Globo 3D")
fig_3d.show()


## 📊 9. Discusión, Limitaciones y Conclusiones del Reto

### 9.1 ¿Qué tan buena es la solución obtenida?
1. **Cobertura Global y Prioridades**:
   - Con solo **8 satélites**, el algoritmo evolutivo logró cubrir las **6 regiones de interés**, garantizando máxima calidad de observación en las zonas de prioridad **Alta** (Península de Yucatán y Amazonía brasileña) y balanceando adecuadamente las de prioridad **Media** (California, Sahel, Japón) y **Normal** (Interior de Australia).
   - El puntaje de cobertura ponderada supera consistentemente el **90-95%**, asegurando que los satélites se ubiquen en ángulos cercanos al nadir sobre los objetivos prioritarios.

2. **Cumplimiento Estricto de Restricciones**:
   - En la Etapa 1 (sin restricciones), el algoritmo descubrió configuraciones que a menudo colisionaban con las zonas de desechos o posiciones ocupadas.
   - En la Etapa 2, gracias a la combinación de una **función de penalización con gradiente de penetración** y el **operador de reparación**, el sistema eliminó el 100% de las violaciones de seguridad sin comprometer la cobertura de las regiones estratégicas.

3. **Capacidad de Adaptación Dinámica**:
   - Al introducir un evento repentino (desechos imprevistos en el Caribe o cambio de prioridades), el algoritmo demostró capacidad de re-planificación rápida, ajustando la posición de los satélites afectados hacia posiciones seguras adyacentes manteniendo la región en su borde de cobertura.

### 9.2 Limitaciones del Modelo Educativo Simplificado
- **Mecánica Orbital Real vs. Instantánea Estática**: El modelo actual evalúa una configuración de sub-puntos satelitales en un instante dado. En una misión orbital real, los satélites siguen trayectorias keplerianas (constelaciones tipo Walker Delta o Walker Star) donde la cobertura es temporalmente periódica con tiempos de revisita y no estática.
- **Tiempos de Maniobra y Combustible ($\Delta V$)**: En un despliegue real, reubicar un satélite ante un cambio dinámico consume propelente y requiere horas o días de maniobras orbitales de Hohmann o de fase.
- **Topografía y Oblicuidad Atmosférica**: El radio de observación de 300 km sobre el terreno asume visibilidad geométrica pura sin atenuación de ángulo de elevación mínimo (típicamente $\ge 10^\circ-15^\circ$ sobre el horizonte para evitar distorsión atmosférica).

---
### 🎓 Estructura de Entregables
Este cuaderno `.ipynb` contiene:
- El modelo matemático riguroso.
- La implementación completa del Algoritmo Genético.
- Las dos etapas de resolución (sin restricciones vs con restricciones).
- Las pruebas de estrés ante cambios dinámicos.
- Visualización cartográfica 2D y el **globo 3D interactivo con Plotly**.
